In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyaml.accelerator import Accelerator
from pyaml.common.constants import Action

## Facility configuration

Edit the `CONFIG_FILE` variable below to select the facility. The rest of the notebook runs unchanged.

| Facility | Config file | Control system | Virtual twin |
|----------|-------------|----------------|--------------|
| SOLEIL II | `config/soleil_ii/p.yaml` | TANGO (`tango-pyaml`) | Apptainer container |
| BESSY2 | `config/bessy2/bessy2.yaml` | EPICS (`pyaml-cs-oa`) | Apptainer container |
| ESRF EBS | `config/esrf/esrf.yaml` | TANGO (`tango-pyaml`) | Apptainer container |

See `README.md` for installation and virtual accelerator startup instructions.

In [2]:
# ── Facility configuration ──────────────────────────────────────────────────
# Uncomment the facility you want to use:

# CONFIG_FILE = "config/soleil_ii/p.yaml"
# CONFIG_FILE = "config/bessy2/bessy2.yaml"
CONFIG_FILE = "config/esrf/esrf.yaml"
# ────────────────────────────────────────────────────────────────────────────

CONFIG_DIR = Path(CONFIG_FILE).parent

sr = Accelerator.load(CONFIG_FILE)
sr  # string representation

PyAML Tango control system binding (0.4.0) initialized with name 'live' and TANGO_HOST=ebs-simu-3:10000


14 Aug 2026, 13:40:30 | WARNING | /home/gubaidulin/codes/operation/pyAML/pyaml/examples/unified_examples/config/esrf/trm.json: file not found
14 Aug 2026, 13:40:30 | WARNING | /home/gubaidulin/codes/operation/pyAML/pyaml/examples/unified_examples/config/esrf/chroma_resp.json: file not found
14 Aug 2026, 13:40:30 | WARNING | Loading /home/gubaidulin/codes/operation/pyAML/pyaml/examples/unified_examples/config/esrf/orm.json failed /home/gubaidulin/codes/operation/pyAML/pyaml/examples/unified_examples/config/esrf/orm.json: file not found


Accelerator(facility='ESRF', machine='sr', description=None, live=TangoControlSystem(name='live', tango_host='ebs-simu-3:10000', catalog=<tango.pyaml.static_catalog.StaticCatalog object at 0x7f29ddb4e1b0>, debug_level=None, lazy_devices=True, timeout_ms=3000), design=Simulator(name='design', lattice='/home/gubaidulin/codes/operation/pyAML/pyaml/examples/unified_examples/config/esrf/sr/lattices/ebs.mat', mat_key=None, linker=None, description=None), yellow_pages=Controls:
    live
    .

Simulators:
    design
    .

Arrays:
    BPM                   (pyaml.arrays.bpm_array)                size=320
    HCorr                 (pyaml.arrays.magnet_array)             size=288
    VCorr                 (pyaml.arrays.magnet_array)             size=288
    Skews                 (pyaml.arrays.magnet_array)             size=288
    Sext                  (pyaml.arrays.magnet_array)             size=180
    QForTune              (pyaml.arrays.magnet_array)             size=124
    .

Tools:
    BB

### Virtual accelerator setup

For **live** control mode you need a running control system or emulator. If you want to skip this, run the notebook in **design** mode only.

**SOLEIL II / ESRF (TANGO)**
```
apptainer pull virtual-accelerator.sif oras://gitlab-registry.synchrotron-soleil.fr/software-control-system/containers/apptainer/virtual-accelerator:latest
apptainer run virtual-accelerator.sif
```

**BESSY2 (EPICS)**
```
apptainer run oras://registry.hzdr.de/digital-twins-for-accelerators/containers/pyat-softioc-digital-twin:default-v0-5-1-bessy.2711893
```

> **Note for BESSY2 live mode:** the PV prefix in `config/bessy2/bessy2.yaml` must match your virtual accelerator instance. Edit the `prefix:` field under `controls:`.

### Control mode choice

- **`sr.design`** — runs pyAT locally, no control system needed. Set `wait_time = 0.0`.
- **`sr.live`** — connects to the real machine or virtual twin. Set `wait_time` to allow readback settling (typically 1.5–2 s).

In [3]:
SR = sr.design
# SR = sr.live

wait_time = 0.0 if SR == sr.design else 2.0
SR  # string representation

Simulator(name='design', lattice='/home/gubaidulin/codes/operation/pyAML/pyaml/examples/unified_examples/config/esrf/sr/lattices/ebs.mat', mat_key=None, linker=None, description=None)

#### Betatron tune monitor

The tune monitor is defined in the configuration file under the name `BETATRON_TUNE`.

In [4]:
tune_monitor = SR.get_betatron_tune_monitor("BETATRON_TUNE")
print(f"Current tune: {tune_monitor.tune.get()}")
tune_monitor  # string representation

Current tune: [0.15999673 0.33999769]


BetatronTuneMonitor(tune_h='srdiag/beam-tune/main/Qh', tune_v='srdiag/beam-tune/main/Qv', name='BETATRON_TUNE')

#### Quadrupolar correctors

The `QForTune` array contains the quadrupoles used for tune correction. You can access and set individual corrector strengths.

In [ ]:
qcorrectors = SR.magnets.get("QForTune")
first_q = qcorrectors[0]
print(f"The ring has {len(qcorrectors)} quadrupolar correctors. First: {first_q.get_name()}")
qcorrectors[0]  # string representation

The ring has 124 quadrupolar correctors. First: QD2E-C04


Quadrupole(peer='Simulator:design', name='QD2E-C04', model_name='QD2E-C04', magnet_model=LinearMagnetModel(curve=CSVCurve(file='sr/magnet_models/QD2_strength.csv'), powerconverter='srmag/vps-qd2/c04-e/current', calibration_factor=0.999305341, calibration_offset=0.0, crosstalk=0.99912, unit='1/m', hardware_unit='A'))

In [6]:
str_before = qcorrectors[0].strength.get()
print(f"Current strength: {qcorrectors[0].strength.get()=:.4f}")
qcorrectors[0].strength.set(str_before + 0.002)
print(f"After stepping by 0.002: {qcorrectors[0].strength.get()=:.4f}")
qcorrectors[0].strength.set(str_before)
print(f"Reset to {str_before:.4f}")

Current strength: qcorrectors[0].strength.get()=-0.6176
After stepping by 0.002: qcorrectors[0].strength.get()=-0.6156
Reset to -0.6176


### Standard tune correction tool

`SR.tune` is the `DEFAULT_TUNE_CORRECTION` tool. `SR.trm` is the `DEFAULT_TUNE_RESPONSE_MATRIX` tool.

Before correcting the tune you need a response matrix. It can be measured (below) or loaded from a previously saved file.

In [7]:
SR.tune  # string representation

Tune(peer='Simulator:design', name='DEFAULT_TUNE_CORRECTION', description=None, lattice_names=None, quad_array_name='QForTune', betatron_tune_name='BETATRON_TUNE', response_matrix=None)

#### Measuring the tune response matrix

The callback below prints progress during the measurement. The `sleep_between_step` parameter controls the wait time between corrector steps — set it to 0 for design mode.

> **Note:** on some lattices `disable_6d()` is required before measuring in design mode.

In [8]:
# Required on some lattices before measuring the TRM in design mode
sr.design.get_lattice().disable_6d()


def tune_callback(action: int, cb_data: dict):
    if action == Action.MEASURE:
        print(f"Tune response: #{cb_data['step']} {cb_data['magnet']} {cb_data['tune']}")
    return True

In [9]:
if SR.tune.response_matrix is None:
    SR.trm.measure(sleep_between_step=wait_time, callback=tune_callback)
    SR.trm.save(CONFIG_DIR / "trm.json")

SR.tune.load(CONFIG_DIR / "trm.json")
print("Response matrix loaded.")
SR.trm  # string representation

Tune response: #0 QD2E-C04 [0.15998216 0.34012592]
Tune response: #0 QD2A-C05 [0.15998212 0.34012589]
Tune response: #0 QD2E-C05 [0.1599821  0.34012584]
Tune response: #0 QD2A-C06 [0.15998217 0.34012594]
Tune response: #0 QD2E-C06 [0.15998208 0.34012584]
Tune response: #0 QD2A-C07 [0.15998213 0.34012602]
Tune response: #0 QD2E-C07 [0.15998215 0.340126  ]
Tune response: #0 QD2A-C08 [0.15998206 0.34012578]
Tune response: #0 QD2E-C08 [0.15998218 0.34012586]
Tune response: #0 QD2A-C09 [0.15998211 0.34012597]
Tune response: #0 QD2E-C09 [0.15998211 0.34012589]
Tune response: #0 QD2A-C10 [0.15998217 0.34012598]
Tune response: #0 QD2E-C10 [0.15998208 0.34012601]
Tune response: #0 QD2A-C11 [0.15998214 0.34012583]
Tune response: #0 QD2E-C11 [0.15998213 0.3401259 ]
Tune response: #0 QD2A-C12 [0.15998208 0.3401259 ]
Tune response: #0 QD2E-C12 [0.15998217 0.34012583]
Tune response: #0 QD2A-C13 [0.1599821  0.34012601]
Tune response: #0 QD2E-C13 [0.15998212 0.34012598]
Tune response: #0 QD2A-C14 [0.1

TuneResponseMatrix(peer='Simulator:design', name='DEFAULT_TUNE_RESPONSE_MATRIX', description=None, lattice_names=None, n_step=1, sleep_between_step=0, n_avg_meas=1, sleep_between_meas=0, quad_array_name='QForTune', betatron_tune_name='BETATRON_TUNE', quad_delta=0.0001)

#### Correcting the tune

`SR.tune.set([qx, qy])` runs the correction iteratively. The `iter` parameter controls the number of iterations and `wait_time` the settling time between each iteration.

In [10]:
print(f"Tune before correction: {SR.tune.readback()}")

qx, qy = 0.19, 0.28
print(f"\nSetting tune to [{qx}, {qy}]")
SR.tune.set([qx, qy], iter=10, wait_time=wait_time)
print(f"Tune after correction: {SR.tune.readback()}")

qx, qy = 0.21, 0.30
print(f"\nSetting tune to [{qx}, {qy}]")
SR.tune.set([qx, qy], iter=10, wait_time=wait_time)
print(f"Tune after correction: {SR.tune.readback()}")

SR.tune  # string representation

Tune before correction: [0.16000001 0.33999986]

Setting tune to [0.19, 0.28]
Tune after correction: [0.19 0.28]

Setting tune to [0.21, 0.3]
Tune after correction: [0.21 0.3 ]


Tune(peer='Simulator:design', name='DEFAULT_TUNE_CORRECTION', description=None, lattice_names=None, quad_array_name='QForTune', betatron_tune_name='BETATRON_TUNE', response_matrix=ResponseMatrixData(matrix=[[0.1785312775665071, 0.17890918518254084, 0.17917983711424057, 0.1784306145899417, 0.17932054383273943, 0.17884878472013144, 0.17868979674900975, 0.17950221390589105, 0.17836418103545082, 0.17902879800274496, 0.17907228791497198, 0.1784534376253477, 0.17936885137875835, 0.1787317009097067, 0.17880339380355048, 0.17934220350290797, 0.17843576159082275, 0.17914061857193797, 0.1789550646144611, 0.17850485174275565, 0.15843662941744663, 0.15753129847057012, 0.15237976934923125, 0.15250702233238211, 0.17830407720298425, 0.17955648207695907, 0.17845839986763146, 0.1788046875300653, 0.17936621410946652, 0.17824945151401206, 0.17941811039923206, 0.179053720016642, 0.17841113460537183, 0.17957284018182973, 0.17835112510161677, 0.17897117554260822, 0.1792322649588063, 0.1782539868933064, 0.17